# OpenRouter model screening before Tinker training

One secret-carrying answer per APPS problem/model, at most **100 shared problems**.
Compare functional, exact-message, and joint pass@1. Inference uses OpenRouter, with a Codex Luna baseline below;
later training would be on Tinker. This notebook does not train models.

**Preparation is free of inference calls.** It saves all requests and estimates cost.
The later execution cell asks yes/no before spending on OpenRouter and Modal.
See [README.md](README.md) for model availability, limitations, and artifact schemas.

In [ ]:
import json
import os
import random
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic_yaml import parse_yaml_raw_as

# Locate the repository when Jupyter starts in this notebook's subdirectory.
repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents)
                 if (path / "ciphers/variable_naming_in_python_v2").is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
from ciphers.variable_naming_in_python_v2.data.apps import REPO_ROOT
from ciphers.variable_naming_in_python_v2.data.codex_apps import SecretTask
from ciphers.variable_naming_in_python_v2.decoder import CipherConfig
from ciphers.variable_naming_in_python_v2.tinker.codex_evaluate import prepare_codex_comparison, run_codex_comparison
from ciphers.variable_naming_in_python_v2.tinker.openrouter_evaluate import run_prepared, summarize
from ciphers.variable_naming_in_python_v2.tinker.openrouter_prepare import RunConfig, artifact_path, prepare_run

if not all(os.environ.get(key) for key in ("OPENROUTER_API_KEY", "STEGO_ARTIFACTS_DIR")):
    env_path = REPO_ROOT / ".env"
    if not env_path.is_file():
        raise FileNotFoundError("Set the environment variables or provide the repository-root .env")
    load_dotenv(env_path, override=False)
for key in ("OPENROUTER_API_KEY", "STEGO_ARTIFACTS_DIR"):
    if not os.environ.get(key):
        raise RuntimeError(f"Missing required environment variable: {key}")

In [ ]:
cipher_path = REPO_ROOT / "ciphers/variable_naming_in_python_v2/tinker/official_cipher.yaml"
secret = SecretTask(
    cipher=parse_yaml_raw_as(CipherConfig, cipher_path.read_text()),
    message_bits="101",
)
config = RunConfig(secret=secret, num_problems=100, max_tokens=16384, estimated_output_tokens=4096, timeout_s=900)
config

## Prepare and review

The next cell downloads the filtered APPS dataset and public model prices, writes all
requests, then costs the saved prompts. It sends **no inference requests**.
To review an existing run, replace the prepare call with `Path("variable_naming_v2/tinker/<run-id>")`.

The estimate assumes 4,096 output tokens per answer; the limit scenario assumes
16,384, including reasoning where the provider includes it in the limit.
Neither is a hard spending ceiling. Modal grading and account fees are excluded.
The shared request deadline is 900 seconds to accommodate slower reasoning answers.

In [ ]:
run_dir = prepare_run(config)
print("Run directory:", artifact_path(run_dir))
estimate = json.loads((artifact_path(run_dir) / "estimate.json").read_text())
price_rows = []
for row in estimate["models"]:
    pricing = row["pricing"]
    price_rows.append({
        "model": row["model"], "available": row["available"], "requests": row["requests"],
        "input_characters": row["input_characters"], "estimated_input_tokens": row["input_tokens"],
        "input_USD_per_M": pricing["prompt"] * 1e6 if pricing else None,
        "output_USD_per_M": pricing["completion"] * 1e6 if pricing else None,
        "estimated_USD": row["estimated_usd"], "limit_scenario_USD": row["limit_scenario_usd"],
    })
display(pd.DataFrame(price_rows))
print(f"Estimated inference cost: ${estimate['estimated_usd']:.2f}")
print(f"Output-limit scenario: ${estimate['limit_scenario_usd']:.2f}")
print("Inspect requests.jsonl, grading_cases.json, and config.json in the run directory.")

## Preview saved prompts

Randomly sample three request rows from the saved `requests.jsonl` file and display
their actual prompts, with model and problem IDs. The seed makes the sample repeatable;
different model rows can contain the same problem. This cell makes no API calls.

In [ ]:
request_rows = [
    json.loads(line)
    for line in (artifact_path(run_dir) / "requests.jsonl").read_text().splitlines()
]
sampled_requests = random.Random(config.seed).sample(request_rows, k=min(3, len(request_rows)))
for request_row in sampled_requests:
    display(Markdown(f"### APPS {request_row['problem_id']} — {request_row['body']['model']}"))
    for message in request_row["body"]["messages"]:
        display(Markdown(message["content"]))

## Optional paid execution

Run only after reviewing the saved requests and estimate. Answer `yes` to start;
anything else sends no inference requests. Execution uses the selected worker count, with one answer
per problem/model, and saves results incrementally. An infrastructure error stops
the run. Started runs cannot be rerun automatically.

In [ ]:
num_workers = 16  # Set to 1 for sequential execution, or 32 for more concurrency.
answer = input(f"Run {sum(row['requests'] for row in estimate['models'])} saved requests? "
               f"Estimate ${estimate['estimated_usd']:.2f}; output-limit scenario "
               f"${estimate['limit_scenario_usd']:.2f}, plus Modal. Type yes/no: ").strip().lower()
if answer == "yes":
    results = run_prepared(run_dir, approved=True, num_workers=num_workers)
else:
    print("No inference requests sent.")

In [ ]:
# Safe to run before inference or after an interrupted run; incomplete rates stay blank.
display(pd.DataFrame(summarize(run_dir)))

## Codex Luna on the same saved inputs

Use the prepared `run_dir` above, even if you skip OpenRouter inference. This
copies one prompt per problem into `codex-luna/`, preserving the exact user text,
cipher, message, and private grading cases. There is no resampling or prompt repair.

Luna uses the existing ChatGPT-backed Codex login with tools disabled and a fresh
thread per answer. The shared threaded runner applies the same output parser,
Modal evaluator, and decoder. Codex receives prompt-driven JSON instructions,
without API schema enforcement.

**Provider differences:** Codex has fixed no-tools base instructions, its own
reasoning/sampling defaults, and no equivalent output-token cap in this helper.
These are recorded in `comparison.json`; only the task inputs and scoring are
strictly matched. Older OpenRouter runs using an earlier cipher are not comparable.

In [ ]:
codex_run_dir = prepare_codex_comparison(run_dir)
codex_num_workers = 16
print("Prepared Luna inputs:", artifact_path(codex_run_dir))

### Run Luna (subscription usage and Modal grading)

Running the next cell explicitly starts the 100-problem Luna comparison. It does
not send OpenRouter requests. Responses and grading results are saved incrementally;
started runs cannot be executed again automatically.

In [ ]:
codex_run_dir = run_codex_comparison(run_dir, approved=True, num_workers=codex_num_workers)
display(pd.DataFrame(summarize(codex_run_dir)))

In [ ]:
# Rates stay blank for models whose matching prepared run has not completed.
display(pd.DataFrame(summarize(run_dir) + summarize(codex_run_dir)))

## Recorded Luna result

Run dated 2026-09-16 UTC: **100 first completed answers**, generated
with **16 workers** for the same 100 APPS problems, **77-group official cipher**,
four length bits, and payload `101`.

| Metric | Result |
| --- | --- |
| All supplied code tests passed | **79 passed, 20 failed, 1 grading error** |
| Exact secret message recovered | **87/100 (87%)** |
| Both conditions | **69/100 (69%)** |

Functional success over all 100 is bounded by **79–80%**;
joint success is exact because the ungraded answer also fails message decoding.
The unresolved answer, APPS 4039, repeatedly kills the
Modal evaluator with exit code 137. Its code constructs lists over extremely large
ranges, suggesting memory exhaustion; the error remains an infrastructure/runner
error under the existing evaluator contract. It is not silently counted as a
failed unit test. `summarize()` leaves rates blank because only 99 grades completed.

The first run stopped with a 300-second SDK timeout after saving 84 grades and 85
answers. Recovery retained all completed answers, retried grading the saved answer,
started 14 previously unclaimed problems, and retried the one SDK call without a
final answer using a 900-second deadline. **101 SDK generation attempts yielded
100 final answers**; no completed answer was regenerated or repaired. There were
**0** malformed/empty/truncated final answers. The notebook
now uses a shared 900-second deadline for future provider comparisons.

Artifacts relative to `STEGO_ARTIFACTS_DIR`:
`variable_naming_v2/tinker/20260915T235628Z-b76630f2/codex-luna-comparison`.
`summary.json` contains counts and bounds; `recovery.json` links the preserved
original and recovery runs; `ungraded_answers.json` records the unresolved grade
and its independently decoded message. The parent directory contains matching
OpenRouter requests; no OpenRouter inference was run for this cipher. Older
OpenRouter results use different cipher settings and are excluded.

Task inputs and scoring are shared. Codex's fixed base instructions, native
sampling/reasoning defaults, and lack of the OpenRouter output-token cap remain
provider differences, so this is a practical screening baseline.


In [ ]:
recorded_codex_run = Path("variable_naming_v2/tinker/20260915T235628Z-b76630f2/codex-luna-comparison")
recorded_report = json.loads((artifact_path(recorded_codex_run) / "summary.json").read_text())
display(recorded_report)